# Conformer — Convolution-augmented Transformer for Speech Recognition (2020)

_Papers · Speech & Audio_

**Put a convolution module inside a Transformer block so one layer sees both the global picture (attention) and fine local detail (convolution).**

---

This notebook is a practice scaffold for the **Conformer — Convolution-augmented Transformer for Speech Recognition (2020)** lesson. We build it up one step at a time: first trace Equation 1 on a single position, then assemble the block from its sub-modules, then train it on a toy local-spike task and ablate the convolution. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build the Conformer block from the inside out in four steps: (1) trace Equation 1's four residual lines on one position, (2) define the sub-modules and the block (with an ablation toggle), (3) make a toy task whose label depends on a *local* spike, and (4) train with and without the convolution module to see what it contributes.

### Step 1 — Trace Equation 1 on one position

A Conformer block is four residual updates (Equation 1): a half-weighted front FFN, full self-attention, a full convolution module, and a half-weighted back FFN, then a final LayerNorm. The two FFNs are *half* residuals (the "Macaron" design), while attention and conv are full residuals. We trace these at a single scalar position so the arithmetic is visible, and note what the pre-LayerNorm value would be if the convolution line were ablated.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Worked example: the four lines of Equation 1 at one position, D=1.
x = 1.0
xt  = x  + 0.5 * 0.4         # line 1, front half-FFN: 1.0 + 0.2 = 1.2
xp  = xt + 0.2               # line 2, attention (full residual):       1.4
xpp = xp + (-0.6)            # line 3, conv (full residual, local):     0.8  -- ablation deletes this
preLN = xpp + 0.5 * 0.5      # line 4, back half-FFN:                   1.05 (then LayerNorm)

print("line1 x~  =", xt)                                       # 1.2
print("line2 x'  =", xp)                                       # 1.4
print("line3 x'' =", xpp)                                      # 0.8
print("pre-LN    =", round(preLN, 3))                          # 1.05
print("pre-LN if conv ABLATED =", round(xp + 0.5 * 0.5, 3))    # 1.65 (the -0.6 local fix is gone)

### Step 2 — Define the sub-modules and the Conformer block

Now we build the real modules. `FFN` is LayerNorm -> Linear(D, 4D) -> Swish -> Linear(4D, D) (Section 2.3); `F.silu` *is* Swish. `ConvModule` (Section 2.2) is the local path: pointwise conv expands channels for a GLU gate, a **depthwise** conv applies one filter per channel over neighbouring frames, then BatchNorm, Swish, and a pointwise conv back. The `ConformerBlock` wires the four residual lines of Equation 1 together, with `use_conv` toggling the convolution-module ablation.

In [ ]:
# The Conformer block. use_conv toggles the convolution-module ablation.
class FFN(nn.Module):                            # Section 2.3: LN -> Linear(D,4D) -> Swish -> Linear(4D,D)
    def __init__(self, D):
        super().__init__()
        self.ln = nn.LayerNorm(D)
        self.up = nn.Linear(D, 4 * D)
        self.down = nn.Linear(4 * D, D)
    def forward(self, x):
        return self.down(F.silu(self.up(self.ln(x))))   # F.silu IS Swish

class ConvModule(nn.Module):                     # Section 2.2 / Figure 2
    def __init__(self, D, k=15):
        super().__init__()
        self.ln = nn.LayerNorm(D)
        self.pw1 = nn.Conv1d(D, 2 * D, 1)        # pointwise, expand to 2D for the GLU
        self.glu = nn.GLU(dim=1)                  # splits 2D channels -> value half gated by sigmoid(other half)
        self.dw = nn.Conv1d(D, D, k, padding=k // 2, groups=D)  # DEPTHWISE: one filter per channel (local)
        self.bn = nn.BatchNorm1d(D)
        self.pw2 = nn.Conv1d(D, D, 1)            # pointwise back to D
    def forward(self, x):                         # x: (B, T, D)
        z = self.ln(x).transpose(1, 2)            # -> (B, D, T) for Conv1d
        z = self.glu(self.pw1(z))                 # (B, D, T)
        z = F.silu(self.bn(self.dw(z)))           # depthwise -> BatchNorm -> Swish
        z = self.pw2(z)
        return z.transpose(1, 2)                   # -> (B, T, D)

class ConformerBlock(nn.Module):
    def __init__(self, D, heads=4, use_conv=True):
        super().__init__()
        self.use_conv = use_conv
        self.ffn1 = FFN(D)
        self.ln_attn = nn.LayerNorm(D)
        self.attn = nn.MultiheadAttention(D, heads, batch_first=True)  # global path
        self.conv = ConvModule(D)                  # local path
        self.ffn2 = FFN(D)
        self.ln_out = nn.LayerNorm(D)
    def forward(self, x):                          # x: (B, T, D), Equation 1
        x = x + 0.5 * self.ffn1(x)                 # line 1: front half-FFN
        a = self.ln_attn(x)
        x = x + self.attn(a, a, a)[0]              # line 2: self-attention (full residual)
        if self.use_conv:
            x = x + self.conv(x)                   # line 3: convolution module -- the ablated line
        x = self.ln_out(x + 0.5 * self.ffn2(x))    # line 4: back half-FFN + final LayerNorm
        return x

### Step 3 — Build a toy task with a local label

The task is deliberately *local*: a short 3-frame spike is placed in the left half or the right half of the sequence, and which half it lands in sets the binary label. A global average (what attention computes) tends to blur such a short pattern, so this task is exactly where a local convolution should help. `Net` pools the block output over time and classifies; `run` trains it and returns held-out accuracy for a given `use_conv` setting.

In [ ]:
# Toy LOCAL-spike task: a short bump placed in the left vs right half sets the label.
def make_data(n=1200, T=40, D=16):
    X = torch.randn(n, T, D) * 0.3
    y = torch.randint(0, 2, (n,))
    for i in range(n):
        if y[i] == 0:
            pos = torch.randint(2, T // 2 - 2, (1,)).item()
        else:
            pos = torch.randint(T // 2 + 2, T - 2, (1,)).item()
        X[i, pos - 1:pos + 2, :] += 2.5           # a short, LOCAL 3-frame spike -> label depends on WHERE it is
    return X, y

Xtr, ytr = make_data(1000)
Xte, yte = make_data(400)

class Net(nn.Module):
    def __init__(self, D=16, use_conv=True):
        super().__init__()
        self.block = ConformerBlock(D, use_conv=use_conv)
        self.head = nn.Linear(D, 2)
    def forward(self, x):
        return self.head(self.block(x).mean(1))    # pool over time, classify

def run(use_conv, epochs=12, lr=3e-3):
    torch.manual_seed(0)
    net = Net(use_conv=use_conv)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for ep in range(epochs):
        net.train()
        opt.zero_grad()
        loss = F.cross_entropy(net(Xtr), ytr)
        loss.backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        acc = (net(Xte).argmax(1) == yte).float().mean().item()
    return acc

### Step 4 — Run the ablation

Everything is held identical between the two runs — depth, width, heads, optimizer, data, and seed — so the only difference is whether line 3 (the convolution module) is present. With the conv module the block reaches high accuracy; ablated, it drops, because attention alone smears the short local spike. This mirrors the paper's Table 3, where removing the convolution is the most damaging change.

In [ ]:
acc_conv = run(use_conv=True)
acc_abl  = run(use_conv=False)

print(f"\nlocal-spike test acc  WITH conv module: {acc_conv:.3f}   ABLATED (no conv): {acc_abl:.3f}")
# WITH conv reaches ~0.97; ABLATED drops to ~0.74 -- the depthwise conv supplied the local view.
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported WER.)

## Visualize the data & results

_On a toy task whose label depends on a short LOCAL spike, does the convolution module help, and does removing it (the Table-3 ablation) lower accuracy because attention alone blurs local patterns?_

### Step 1 — Rebuild the block and toy data

The visualization re-runs the experiment self-contained so it can plot the *learning curve* with and without the convolution module. We redefine the same `FFN`, `ConvModule`, `ConformerBlock`, and the local-spike `make_data`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class FFN(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.ln = nn.LayerNorm(D)
        self.up = nn.Linear(D, 4 * D)
        self.down = nn.Linear(4 * D, D)
    def forward(self, x):
        return self.down(F.silu(self.up(self.ln(x))))

class ConvModule(nn.Module):
    def __init__(self, D, k=15):
        super().__init__()
        self.ln = nn.LayerNorm(D)
        self.pw1 = nn.Conv1d(D, 2 * D, 1)
        self.glu = nn.GLU(dim=1)
        self.dw = nn.Conv1d(D, D, k, padding=k // 2, groups=D)
        self.bn = nn.BatchNorm1d(D)
        self.pw2 = nn.Conv1d(D, D, 1)
    def forward(self, x):
        z = self.ln(x).transpose(1, 2)
        z = self.glu(self.pw1(z))
        z = F.silu(self.bn(self.dw(z)))
        return self.pw2(z).transpose(1, 2)

class ConformerBlock(nn.Module):
    def __init__(self, D, heads=4, use_conv=True):
        super().__init__()
        self.use_conv = use_conv
        self.ffn1 = FFN(D)
        self.ln_attn = nn.LayerNorm(D)
        self.attn = nn.MultiheadAttention(D, heads, batch_first=True)
        self.conv = ConvModule(D)
        self.ffn2 = FFN(D)
        self.ln_out = nn.LayerNorm(D)
    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)
        a = self.ln_attn(x)
        x = x + self.attn(a, a, a)[0]
        if self.use_conv:
            x = x + self.conv(x)               # the ablated line (Eq 1, line 3)
        return self.ln_out(x + 0.5 * self.ffn2(x))

def make_data(n, T=40, D=16):
    X = torch.randn(n, T, D) * 0.3
    y = torch.randint(0, 2, (n,))
    for i in range(n):
        if y[i] == 0:
            pos = torch.randint(2, T // 2 - 2, (1,)).item()
        else:
            pos = torch.randint(T // 2 + 2, T - 2, (1,)).item()
        X[i, pos - 1:pos + 2, :] += 2.5
    return X, y

Xtr, ytr = make_data(1000)
Xte, yte = make_data(400)

### Step 2 — Train and record accuracy per epoch

This `run` is like the reference one but appends test accuracy after *every* epoch, returning the full history so we can watch the curves. `Net` again pools over time and classifies.

In [ ]:
class Net(nn.Module):
    def __init__(self, D=16, use_conv=True):
        super().__init__()
        self.block = ConformerBlock(D, use_conv=use_conv)
        self.head = nn.Linear(D, 2)
    def forward(self, x):
        return self.head(self.block(x).mean(1))

def run(use_conv, epochs=12):
    torch.manual_seed(0)
    net = Net(use_conv=use_conv)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    hist = []
    for ep in range(epochs):
        net.train()
        opt.zero_grad()
        F.cross_entropy(net(Xtr), ytr).backward()
        opt.step()
        net.eval()
        with torch.no_grad():
            acc = (net(Xte).argmax(1) == yte).float().mean().item()
            hist.append(round(acc, 3))
    return hist

### Step 3 — Compare the two learning curves

Printing both histories shows the gap: with the convolution the curve climbs to high accuracy, while without it the curve plateaus low — attention alone keeps blurring the local spike across all epochs.

In [ ]:
print("conv on :", run(True))
print("conv off:", run(False))
# conv on -> climbs to ~0.97. conv off -> plateaus ~0.74 (attention alone blurs the local spike).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your Conformer block solves a toy task whose label depends on a short local
            spike in the sequence. Remove the convolution module (delete the "$x = x + \mathrm{conv}(x)$" line,
            keeping attention and both half-FFNs) and retrain. What happens to accuracy on the local task, and what
            does it demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Delete only line 3 of Equation 1 &mdash; the conv residual &mdash; and keep depth, width, heads, optimizer, data, and seed identical. — _An honest ablation changes exactly one thing (the convolution module) so any gap is attributable to it._
- Retrain and compare test accuracy: with the conv module ON it is higher; with it OFF it drops (in our run from ~0.97 to ~0.74). — _The task hinges on a local pattern; attention is a global weighted average that blurs short spikes, so without the depthwise conv's local filter the block sees the spike less sharply (&sect;1)._
- Conclude that the convolution module supplies the local modeling, mirroring Table 3 where removing it is the most damaging change. — _Same architecture and parameter budget otherwise; only the conv line differs, isolating it as the cause._

**Answer:** With the convolution module removed, test accuracy on the local-spike task drops (in our run
                 ~0.97 &rarr; ~0.74). Attention alone is a global weighted average and tends to smear short, local
                 patterns, so the block loses the sharp local view the depthwise convolution provided. Because the
                 two runs are identical except for the "$+\,\mathrm{Conv}$" line, this isolates the convolution
                 module as the source of local modeling &mdash; matching the paper, which calls it the most important
                 sub-block (test-other WER $4.3\% \rightarrow 4.9\%$, &sect;3.4 Table 3). The CODEVIZ panel shows the gap.

</details>

**Problem 2.** Trace Equation 1 by hand for one position with state $x = 2.0$, given residual outputs
            $\mathrm{FFN}_1 = 0.6$, $\mathrm{MHSA} = -0.4$, $\mathrm{Conv} = 0.8$, $\mathrm{FFN}_2 = 0.2$
            (ignore the final LayerNorm). What is the pre-LayerNorm value, and what would it be with the conv line
            ablated?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Line 1 (half-FFN): $\tilde{x} = 2.0 + \tfrac{1}{2}(0.6) = 2.3$. — _The front FFN adds only half its output (Macaron, line 1)._
- Line 2 (attention): $x' = 2.3 + (-0.4) = 1.9$. Line 3 (conv): $x'' = 1.9 + 0.8 = 2.7$. — _Attention and conv are full residuals (no half)._
- Line 4 (half-FFN): pre-LN $= 2.7 + \tfrac{1}{2}(0.2) = 2.8$. Ablated (skip line 3): $1.9 + \tfrac{1}{2}(0.2) = 2.0$. — _Removing the conv residual drops the $+0.8$ local correction, changing $2.8$ to $2.0$._

**Answer:** Pre-LayerNorm value $= 2.8$. The two half-FFNs together contributed $0.3 + 0.1 = 0.4$ (what one
                 full FFN would add). With the convolution line ablated the local correction $+0.8$ disappears and
                 the pre-LayerNorm value falls to $2.0$ &mdash; the same arithmetic the worked example walks through.

</details>

**Problem 3.** Why does a single Conformer block model both global and local structure, where a plain
            Transformer block models mostly global? Answer in terms of which line does what (&sect;1, &sect;2).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the global path: line 2, multi-headed self-attention, lets every frame attend to every other &mdash; long-range, global context. — _Attention is the Transformer's strength; the paper notes it is "good at modeling long-range global context" (&sect;1)._
- Identify the local path: line 3, the convolution module, whose depthwise conv slides a small filter over neighbouring frames &mdash; fine-grained local patterns. — _Convolutions "exploit local information"; the paper inserts the conv module precisely to add what attention lacks (&sect;1)._
- Conclude the block sees the sequence both ways in one layer, where a plain Transformer block (attention + FFN only) leans global and is "less capable to extract fine-grained local feature patterns" (&sect;1). — _Composing the complementary strengths in one block is the paper's core contribution (&sect;2)._

**Answer:** Line 2 (self-attention) is the global path &mdash; every frame attends to every other, capturing
                 long-range context. Line 3 (the convolution module, via its depthwise conv) is the local path &mdash;
                 a small sliding filter over neighbouring frames captures fine-grained local patterns. A plain
                 Transformer block has only attention + FFN, so it is strong globally but, in the paper's words,
                 "less capable to extract fine-grained local feature patterns" (&sect;1). The Conformer block has
                 both in one layer, which is exactly its contribution (&sect;2).

</details>